# Threshold calibration on a small target-domain sample (DFD and Celeb-DF v2)

Why: on both datasets the models label most videos as fake at the fixed threshold 0.5, so real videos are often flagged. ROC-AUC (ranking quality) is unaffected by the threshold, but accuracy and balanced accuracy depend on it.

What this notebook does
1. For each dataset it picks a **small calibration set** (default 40 real + 40 fake) that is **disjoint from the test sample** already evaluated.
2. It chooses one decision threshold per model on the calibration set only (maximum balanced accuracy).
3. It applies that threshold to the **untouched test sample** and compares it with the fixed threshold 0.5.

No model is retrained. This is **not zero-shot**: report it as "threshold calibrated on a small sample from the target dataset". The test sample is never used to choose anything.

Before running: you need the saved test predictions on Drive. DFD: `cross_DFD_standalone_branches_predictions.csv` (you have it). Celeb-DF: run the earlier Celeb-DF notebook once with `N_PER_CLASS = 100` (it reuses the videos already cached) so that `cross_CelebDFv2_standalone_branches_predictions.csv` exists; otherwise only the gated model is calibrated for Celeb-DF.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

!pip -q install mediapipe kagglehub scikit-learn
!wget -q -nc https://storage.googleapis.com/mediapipe-models/face_detector/blaze_face_short_range/float16/1/blaze_face_short_range.tflite -O /content/blaze_face_short_range.tflite
print("Setup complete.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Setup complete.


In [4]:
import os, shutil, json, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet18
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision

P = "/content/drive/MyDrive/Multimedia_Authentication"
MODELS_DIR = f"{P}/v2/models"
CKPTS = {
    "gated":     f"{MODELS_DIR}/gated_v2_best.pth",
    "spatial":   f"{MODELS_DIR}/spatial_resnet18_v2_best.pth",
    "frequency": f"{MODELS_DIR}/frequency_resnet18_v2_best.pth",
}
OUT = f"{P}/v2/results"
FIG = f"{OUT}/figs"
FACE_CACHE = f"{P}/cross_dataset_cache/faces_dfd"
for d in (OUT, FIG, FACE_CACHE):
    os.makedirs(d, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
for name, path in CKPTS.items():
    print(f"{name:10s} checkpoint exists:", os.path.exists(path))
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing checkpoint: {path}. Update CKPTS to the exact file.")
if device.type != "cuda":
    warnings.warn("GPU not detected. This will be slow on CPU.")

Device: cpu
gated      checkpoint exists: True
spatial    checkpoint exists: True
frequency  checkpoint exists: True


/tmp/ipykernel_2209/664206399.py:39: UserWarning: GPU not detected. This will be slow on CPU.
  warnings.warn("GPU not detected. This will be slow on CPU.")


## Face extraction and models (same as the evaluation notebooks)

In [5]:
def make_detector():
    options = vision.FaceDetectorOptions(
        base_options=mp_python.BaseOptions(model_asset_path="/content/blaze_face_short_range.tflite"),
        min_detection_confidence=0.5
    )
    return vision.FaceDetector.create_from_options(options)


def read_frames(video_path, n=8):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return None
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        return None
    frames = []
    for frame_idx in np.linspace(0, total - 1, n, dtype=int):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
        ok, frame = cap.read()
        if not ok:
            cap.release()
            return None
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()
    return frames


def detect_box(detector, frame, max_side=640):
    height, width = frame.shape[:2]
    scale = min(1.0, max_side / max(height, width))
    small = (cv2.resize(frame, (int(width * scale), int(height * scale)), interpolation=cv2.INTER_AREA)
             if scale < 1.0 else frame)
    result = detector.detect(mp.Image(image_format=mp.ImageFormat.SRGB,
                                      data=np.ascontiguousarray(small, dtype=np.uint8)))
    if not result.detections:
        return None
    box = max(result.detections, key=lambda d: d.bounding_box.width * d.bounding_box.height).bounding_box
    return (box.origin_x / scale, box.origin_y / scale, box.width / scale, box.height / scale)


def square_crop(frame, box, scale=1.4, out=224):
    height, width = frame.shape[:2]
    x, y, w, h = box
    cx, cy = x + w / 2, y + h / 2
    side = int(min(max(w, h) * scale, height, width))
    side = max(side, 2)
    x1 = int(np.clip(cx - side / 2, 0, width - side))
    y1 = int(np.clip(cy - side / 2, 0, height - side))
    crop = frame[y1:y1 + side, x1:x1 + side]
    if crop.size == 0:
        raise ValueError("Empty face crop")
    return cv2.resize(crop, (out, out), interpolation=cv2.INTER_AREA)


def process_video(detector, video_path):
    frames = read_frames(video_path)
    if frames is None:
        return None, "read_failed"
    boxes = [detect_box(detector, frame) for frame in frames]
    found = [i for i, b in enumerate(boxes) if b is not None]
    if not found:
        return None, "no_face"
    for i, b in enumerate(boxes):
        if b is None:
            boxes[i] = boxes[min(found, key=lambda j: abs(j - i))]
    crops = [square_crop(f, b) for f, b in zip(frames, boxes)]
    return np.stack(crops).astype(np.uint8), "ok"

In [6]:
MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
STD  = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)


def haar_highfreq(x):
    a = x[..., 0::2, 0::2]; b = x[..., 0::2, 1::2]
    c = x[..., 1::2, 0::2]; d = x[..., 1::2, 1::2]
    lh = (a + b - c - d) / 2
    hl = (a - b + c - d) / 2
    hh = (a - b - c + d) / 2
    return torch.stack([lh.mean(1), hl.mean(1), hh.mean(1)], dim=1)


def to_branch(x, branch):
    # x: [B, T, 3, H, W] in 0..1
    B, T, C, H, W = x.shape
    x = x.reshape(B * T, C, H, W)
    if branch == "spatial":
        x = (x - MEAN) / STD
    else:
        h = haar_highfreq(x)
        mn = h.amin(dim=(1, 2, 3), keepdim=True)
        mx = h.amax(dim=(1, 2, 3), keepdim=True)
        h = (h - mn) / (mx - mn + 1e-8)
        h = F.interpolate(h, size=(H, W), mode="bilinear", align_corners=False)
        x = (h - MEAN) / STD
    return x.reshape(B, T, C, H, W)


def backbone():
    return nn.Sequential(*list(resnet18(weights=None).children())[:-1])


class BranchNet(nn.Module):
    # Standalone single-branch model (same layout as training: backbone.*, classifier.*)
    def __init__(self):
        super().__init__()
        self.backbone = backbone()
        self.classifier = nn.Linear(512, 2)

    def forward(self, x):
        B, T = x.shape[:2]
        f = self.backbone(x.flatten(0, 1)).flatten(1)
        return self.classifier(f.view(B, T, 512).mean(1))


class GatedFusionNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.spatial_bb = backbone()
        self.freq_bb = backbone()
        self.gate = nn.Sequential(nn.Linear(1024, 128), nn.ReLU(), nn.Dropout(0.2), nn.Linear(128, 1))
        self.classifier = nn.Sequential(nn.Linear(512, 256), nn.ReLU(), nn.Dropout(0.5), nn.Linear(256, 2))

    def feats(self, x, bb):
        B, T = x.shape[:2]
        return bb(x.flatten(0, 1)).flatten(1).view(B, T, 512).mean(1)


def load(model, path):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    model.load_state_dict(ck["model_state_dict"], strict=True)
    model.to(device).eval()
    print(f"loaded {Path(path).name} | val AUC in checkpoint: {ck.get('val_auc')}")
    return model


gated = load(GatedFusionNet(), CKPTS["gated"])
spatial_net = load(BranchNet(), CKPTS["spatial"])
freq_net = load(BranchNet(), CKPTS["frequency"])


def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


@torch.no_grad()
def predict_all(frames):
    x = (torch.from_numpy(frames).to(device).unsqueeze(0)
         .permute(0, 1, 4, 2, 3).float().div(255.0))
    with torch.autocast("cuda", dtype=torch.float16, enabled=(device.type == "cuda")):
        xs, xf = to_branch(x, "spatial"), to_branch(x, "frequency")

        # gated model: compute branch features once, then mix
        s = gated.feats(xs, gated.spatial_bb)
        f = gated.feats(xf, gated.freq_bb)
        alpha = torch.sigmoid(gated.gate(torch.cat([s, f], 1)).float())

        def mix(a):
            a = a.to(s.dtype)
            return gated.classifier(a * s + (1 - a) * f)

        lg_gated = mix(alpha)
        lg_a1 = mix(torch.full_like(alpha, 1.0))
        lg_a05 = mix(torch.full_like(alpha, 0.5))
        lg_a0 = mix(torch.full_like(alpha, 0.0))

        # separately trained standalone branches
        lg_s = spatial_net(xs)
        lg_f = freq_net(xf)

    pf = lambda lg: float(torch.softmax(lg.float(), 1)[0, 1])
    p_s, p_f = pf(lg_s), pf(lg_f)
    p_late = float(1 / (1 + np.exp(-(logit(p_s) + logit(p_f)) / 2)))
    return {
        "p_gated": pf(lg_gated), "alpha": float(alpha[0, 0]),
        "p_spatial": p_s, "p_frequency": p_f, "p_late": p_late,
        "p_alpha1": pf(lg_a1), "p_alpha05": pf(lg_a05), "p_alpha0": pf(lg_a0),
    }

loaded gated_v2_best.pth | val AUC in checkpoint: 0.9863362351569898
loaded spatial_resnet18_v2_best.pth | val AUC in checkpoint: 0.9871057040868362
loaded frequency_resnet18_v2_best.pth | val AUC in checkpoint: 0.9649595687331536


## Configuration and helpers

In [7]:
import time
import kagglehub

N_CAL = 40            # calibration videos per class, per dataset
CAL_SEED = 7          # different from the test-sample seed (42)

HANDLES = {
    "DFD": "sanikatiwarekar/deep-fake-detection-dfd-entire-original-dataset",
    "CelebDF": "reubensuju/celeb-df-v2",
}
ROOTS = {"DFD": f"{P}/cross_dataset_cache/DFD", "CelebDF": f"{P}/cross_dataset_cache/Celeb-DF-v2"}
FACE_DIRS = {"DFD": f"{P}/cross_dataset_cache/faces_dfd", "CelebDF": f"{P}/cross_dataset_cache/faces_celebdf"}
for d in FACE_DIRS.values():
    os.makedirs(d, exist_ok=True)

TEST_CSVS = {
    "DFD": [f"{OUT}/cross_DFD_standalone_branches_predictions.csv"],
    "CelebDF": [f"{OUT}/cross_CelebDFv2_standalone_branches_predictions.csv",
                f"{OUT}/cross_CelebDFv2_test_sample_predictions.csv"],
}
MODEL_COLS = {"Gated fusion": "p_gated", "Spatial branch": "p_spatial",
              "Frequency branch": "p_frequency", "Late fusion": "p_late"}


def download_one(handle, candidates, local_path):
    if os.path.isfile(local_path) and os.path.getsize(local_path) > 0:
        return True
    os.makedirs(os.path.dirname(local_path), exist_ok=True)
    last = None
    for cand in candidates:
        for attempt in range(4):
            try:
                res = Path(str(kagglehub.dataset_download(handle, path=cand)))
                if res.is_dir():
                    found = list(res.rglob(os.path.basename(local_path)))
                    if len(found) != 1:
                        raise FileNotFoundError(cand)
                    res = found[0]
                shutil.copy2(res, local_path)
                if os.path.abspath(res) != os.path.abspath(local_path):
                    try:
                        os.remove(res)
                    except OSError:
                        pass
                return True
            except Exception as exc:
                last = exc
                time.sleep(2 * (attempt + 1))
    print("FAILED:", candidates[0], repr(last)[:150])
    return False


def candidates_for(dataset, rel):
    if dataset == "DFD":
        return [rel]
    if rel.startswith("celeb-df-v2/"):
        return [rel, rel[len("celeb-df-v2/"):]]
    return [f"celeb-df-v2/{rel}", rel]


def build_manifest(dataset):
    if dataset == "DFD":
        names = pd.read_csv(f"{ROOTS['DFD']}/dfd_file_list.csv")["name"].tolist()
        rows = []
        for n in names:
            if not n.lower().endswith(".mp4"):
                continue
            low = n.lower()
            label = "FAKE" if "manipulated" in low else ("REAL" if "original" in low else None)
            if label:
                rows.append({"relative_path": n, "label": label})
        return pd.DataFrame(rows).drop_duplicates("relative_path").reset_index(drop=True)
    # Celeb-DF: official test list (1 = REAL, 0 = FAKE)
    rows = []
    with open(f"{ROOTS['CelebDF']}/List_of_testing_videos.txt", encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            flag, rel = line.split(maxsplit=1)
            rows.append({"relative_path": rel.strip().lstrip("/"),
                         "label": "REAL" if flag == "1" else "FAKE"})
    return pd.DataFrame(rows).drop_duplicates("relative_path").reset_index(drop=True)


def to_rel(path):
    # saved CSVs hold either the relative path or a full Drive path
    return path.split("Celeb-DF-v2/", 1)[-1] if "Celeb-DF-v2/" in path else path


def load_test(dataset):
    for path in TEST_CSVS[dataset]:
        if os.path.isfile(path):
            df = pd.read_csv(path)
            if "prob_fake" in df.columns and "p_gated" not in df.columns:
                df = df.rename(columns={"prob_fake": "p_gated"})
            df["rel"] = df["path"].map(to_rel)
            print(f"{dataset}: test predictions from {Path(path).name} ({(df['status'] == 'ok').sum()} usable videos)")
            return df
    raise FileNotFoundError(f"No saved test predictions found for {dataset}: {TEST_CSVS[dataset]}")


def run_inference(dataset, sel):
    root, face_dir = ROOTS[dataset], FACE_DIRS[dataset]
    detector = make_detector()
    rows = []
    for row in tqdm(sel.itertuples(index=False), total=len(sel), desc=f"{dataset} calibration"):
        rel, label = row.relative_path, row.label
        local = os.path.join(root, rel)
        if not download_one(HANDLES[dataset], candidates_for(dataset, rel), local):
            rows.append({"rel": rel, "label": label, "status": "download_failed"})
            continue
        cache = os.path.join(face_dir, rel.replace("/", "__").replace("\\", "__") + ".npy")
        try:
            if os.path.isfile(cache):
                faces, status = np.load(cache), "ok"
            else:
                faces, status = process_video(detector, local)
                if faces is not None:
                    np.save(cache, faces)
            if faces is None:
                rows.append({"rel": rel, "label": label, "status": status})
                continue
            rows.append({"rel": rel, "label": label, "status": "ok", **predict_all(faces)})
        except Exception as exc:
            rows.append({"rel": rel, "label": label, "status": f"error: {type(exc).__name__}"})
    return pd.DataFrame(rows)

## Build the calibration sets and run the models on them
The calibration videos are drawn from videos **not** in the test sample (checked below). If the face detector finds no face in a video, it is skipped, as in the test evaluation.

In [8]:
cal_preds, test_preds = {}, {}
for dataset in ["DFD", "CelebDF"]:
    manifest = build_manifest(dataset)
    test_df = load_test(dataset)
    test_rels = set(test_df["rel"])
    pool = manifest[~manifest["relative_path"].isin(test_rels)]

    rng = np.random.default_rng(CAL_SEED)
    parts = []
    for label in ["REAL", "FAKE"]:
        g = pool[pool["label"] == label]
        idx = rng.choice(g.index.to_numpy(), size=min(N_CAL, len(g)), replace=False)
        parts.append(g.loc[idx])
    sel = pd.concat(parts).reset_index(drop=True)
    assert not (set(sel["relative_path"]) & test_rels), "calibration and test videos must be disjoint"
    print(f"{dataset}: {len(sel)} calibration videos selected ({(sel.label == 'REAL').sum()} real, {(sel.label == 'FAKE').sum()} fake), disjoint from the {len(test_rels)} test videos")

    cal = run_inference(dataset, sel)
    cal.to_csv(f"{OUT}/calibration_{dataset}_predictions.csv", index=False)
    ok = cal[cal["status"] == "ok"]
    print(f"{dataset}: usable calibration videos: {len(ok)} ({(ok.label == 'REAL').sum()} real, {(ok.label == 'FAKE').sum()} fake)\n")
    cal_preds[dataset] = ok
    test_preds[dataset] = test_df[test_df["status"] == "ok"].copy()

DFD: test predictions from cross_DFD_standalone_branches_predictions.csv (174 usable videos)
DFD: 80 calibration videos selected (40 real, 40 fake), disjoint from the 200 test videos


DFD calibration:   0%|          | 0/80 [00:00<?, ?it/s]


  0%|          | 0.00/7.15M [00:00<?, ?B/s]
 14%|█▍        | 1.00M/7.15M [00:01<00:08, 779kB/s]
 28%|██▊       | 2.00M/7.15M [00:01<00:03, 1.52MB/s]
 42%|████▏     | 3.00M/7.15M [00:01<00:01, 2.41MB/s]
 70%|██████▉   | 5.00M/7.15M [00:01<00:00, 4.43MB/s]
100%|██████████| 7.15M/7.15M [00:01<00:00, 3.76MB/s]



  0%|          | 0.00/6.94M [00:00<?, ?B/s]
 14%|█▍        | 1.00M/6.94M [00:01<00:08, 774kB/s]
 29%|██▉       | 2.00M/6.94M [00:01<00:03, 1.51MB/s]
 43%|████▎     | 3.00M/6.94M [00:01<00:01, 2.40MB/s]
 72%|███████▏  | 5.00M/6.94M [00:01<00:00, 4.41MB/s]
100%|██████████| 6.94M/6.94M [00:01<00:00, 3.65MB/s]



  0%|          | 0.00/3.11M [00:00<?, ?B/s]
 32%|███▏      | 1.00M/3.11M [00:01<00:02, 785kB/s]
 64%|██████▍   | 2.00M/3.11M [00:01<00:00, 1.53MB/s]
100%|██████████| 3.11M/3.11M [00:01<00:00, 1.91MB/s]



  0%|          | 0.00/3.98M [00:00<?, ?B/s]
 25%|██▌       | 1.00M/3.98M [00:01<00:04, 782kB/s]
 50%|█████     | 2.00M/3.98M [00:01<00:01, 1.53MB/s]
100%|██████████| 3.98M/3.98M [00:01<00:00, 2.33MB/s]


... (81 outputs left out)

Extracting zip of 00280.mp4...



  0%|          | 0.00/1.21M [00:00<?, ?B/s]
100%|██████████| 1.21M/1.21M [00:01<00:00, 1.24MB/s]

Extracting zip of id28_0004.mp4...


Using Colab cache for faster access to the 'celeb-df-v2' dataset.
Using Colab cache for faster access to the 'celeb-df-v2' dataset.
Using Colab cache for faster access to the 'celeb-df-v2' dataset.
Using Colab cache for faster access to the 'celeb-df-v2' dataset.
Using Colab cache for faster access to the 'celeb-df-v2' dataset.
Using Colab cache for faster access to the 'celeb-df-v2' dataset.
Using Colab cache for faster access to the 'celeb-df-v2' dataset.
Using Colab cache for faster access to the 'celeb-df-v2' dataset.
Using Colab cache for faster access to the 'celeb-df-v2' dataset.
Using Colab cache for faster access to the 'celeb-df-v2' dataset.
Using Colab cache for faster access to the 'celeb-df-v2' dataset.
Using Colab cache for faster access to the 'celeb-df-v2' dataset.
Using Colab cache for faster access to the 'celeb-df-v2' dataset.
Using Colab cache for faster access to the 'celeb-df-v2' dataset.
Using Colab cache for faster access to the 'celeb-df-v2' dataset.
Using Cola

## Choose thresholds on the calibration sets and evaluate on the untouched test samples
The threshold is the value (0.05 to 0.95) with the highest balanced accuracy on the calibration set; ties are broken by the median of the tied values. The `thr IQR` column shows how much the chosen threshold moves when the small calibration set is resampled: a wide range means it is not reliable. Test intervals come from resampling the test videos only.

In [9]:
GRID = np.round(np.arange(0.05, 0.951, 0.01), 2)
rng_eval = np.random.default_rng(42)


def bal_acc(y, p, t):
    pred = p >= t
    return 0.5 * (pred[y == 1].mean() + (~pred[y == 0]).mean())


def pick_threshold(y, p):
    scores = np.array([bal_acc(y, p, t) for t in GRID])
    best = scores.max()
    return float(np.median(GRID[np.isclose(scores, best)])), float(best)


def boot_ci(y, p, t, B=1000):
    real_i, fake_i = np.flatnonzero(y == 0), np.flatnonzero(y == 1)
    vals = []
    for _ in range(B):
        i = np.concatenate([rng_eval.choice(real_i, len(real_i)), rng_eval.choice(fake_i, len(fake_i))])
        vals.append(bal_acc(y[i], p[i], t))
    return np.percentile(vals, [2.5, 97.5])


def thr_spread(y, p, B=200):
    real_i, fake_i = np.flatnonzero(y == 0), np.flatnonzero(y == 1)
    ts = []
    for _ in range(B):
        i = np.concatenate([rng_eval.choice(real_i, len(real_i)), rng_eval.choice(fake_i, len(fake_i))])
        ts.append(pick_threshold(y[i], p[i])[0])
    return np.percentile(ts, [25, 75])


all_rows = []
for dataset in ["DFD", "CelebDF"]:
    cal, test = cal_preds[dataset], test_preds[dataset]
    yc = (cal["label"].values == "FAKE").astype(int)
    yt = (test["label"].values == "FAKE").astype(int)
    for name, col in MODEL_COLS.items():
        if col not in test.columns or col not in cal.columns:
            print(f"{dataset}: skipping {name} (no test predictions saved for it)")
            continue
        pc, pt = cal[col].to_numpy(), test[col].to_numpy()
        thr, cal_bal = pick_threshold(yc, pc)
        q1, q3 = thr_spread(yc, pc)
        lo, hi = boot_ci(yt, pt, thr)
        pred = pt >= thr
        all_rows.append({
            "dataset": dataset, "model": name,
            "test AUC": round(roc_auc_score(yt, pt), 4),
            "bal acc @0.5": round(bal_acc(yt, pt, 0.5), 4),
            "thr (calibrated)": round(thr, 2),
            "thr IQR": f"{q1:.2f}-{q3:.2f}",
            "calib bal acc": round(cal_bal, 3),
            "bal acc @cal thr": round(bal_acc(yt, pt, thr), 4),
            "95% CI": f"{lo:.3f}-{hi:.3f}",
            "real recall": round((~pred[yt == 0]).mean(), 3),
            "fake recall": round(pred[yt == 1].mean(), 3),
            "accuracy": round((pred == yt).mean(), 4),
            "n test": len(yt),
        })

results = pd.DataFrame(all_rows)
pd.set_option("display.width", 250)
print(results.to_string(index=False))
results.to_csv(f"{OUT}/threshold_calibration_results.csv", index=False)
print("\nSaved:", f"{OUT}/threshold_calibration_results.csv")

CelebDF: skipping Spatial branch (no test predictions saved for it)
CelebDF: skipping Frequency branch (no test predictions saved for it)
CelebDF: skipping Late fusion (no test predictions saved for it)
dataset            model  test AUC  bal acc @0.5  thr (calibrated)   thr IQR  calib bal acc  bal acc @cal thr      95% CI  real recall  fake recall  accuracy  n test
    DFD     Gated fusion    0.7626        0.6155              0.93 0.91-0.93          0.735            0.6595 0.585-0.729        0.619        0.700    0.6609     174
    DFD   Spatial branch    0.7520        0.6583              0.68 0.66-0.68          0.734            0.6782 0.602-0.746        0.679        0.678    0.6782     174
    DFD Frequency branch    0.8025        0.7048              0.62 0.62-0.70          0.730            0.7079 0.645-0.773        0.571        0.844    0.7126     174
    DFD      Late fusion    0.8025        0.7167              0.69 0.60-0.75          0.735            0.6770 0.606-0.745        0.64